In [56]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from datasets import load_dataset
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error
import torch
from torch import nn

In [57]:
class _TCNNet(nn.Module):
    def __init__(self, n_features, channels=(32, 64), kernel_size=3, dropout=0.1):
        super().__init__()

        layers = []
        in_ch = n_features

        for i, out_ch in enumerate(channels):
            layers.append(
                TemporalBlock(
                    in_ch,
                    out_ch,
                    kernel_size,
                    dilation=2**i,
                    dropout=dropout
                )
            )
            in_ch = out_ch

        self.tcn = nn.Sequential(*layers)
        self.head = nn.Linear(in_ch, 1)

    def forward(self, x):
        x = x.transpose(1, 2)  # (B, F, T)
        y = self.tcn(x)
        return self.head(y[:, :, -1])

In [59]:
class TCNRegressor:

    def __init__(
        self,
        seq_len=30,
        lr=1e-3,
        epochs=50,
        batch_size=32,
        loss="mse",
        strategy="recursive"
    ):
        self.seq_len = seq_len
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.loss = loss
        self.strategy = strategy

        self.model = None

    # SEQUENCES
    def _create_sequences(self, X, y):

        X = X.values if hasattr(X, "values") else X
        y = y.values if hasattr(y, "values") else y

        Xs, ys = [], []

        for i in range(len(X) - self.seq_len):
            Xs.append(X[i:i + self.seq_len])
            ys.append(y[i + self.seq_len])

        return np.array(Xs), np.array(ys).reshape(-1, 1)

    # FIT
    def fit(self, model, X_train, y_train, X_val=None, y_val=None):

        model.compile(
            optimizer=tf.keras.optimizers.Adam(self.lr),
            loss=self.loss
        )

        X_train_seq, y_train_seq = self._create_sequences(X_train, y_train)

        if X_val is not None:
            X_val_seq, y_val_seq = self._create_sequences(X_val, y_val)
            val_data = (X_val_seq, y_val_seq)
        else:
            val_data = None

        model.fit(
            X_train_seq,
            y_train_seq,
            validation_data=val_data,
            epochs=self.epochs,
            batch_size=self.batch_size,
            shuffle=False
        )

        self.model = model
        return self


    # PREDICT (2 MODES)
    def predict(self, X_last, n_steps=1):

        if self.strategy == "recursive":
            return self._predict_recursive(X_last, n_steps)

        elif self.strategy == "direct":
            return self._predict_direct(X_last)

        else:
            raise ValueError("strategy must be 'recursive' or 'direct'")

    # RECURSIVE (1-step model)
    def _predict_recursive(self, X_last, n_steps):

        X_window = X_last.copy()
        preds = []

        for _ in range(n_steps):

            y = self.model.predict(X_window, verbose=0)[0, 0]
            preds.append(y)

            X_window = np.roll(X_window, -1, axis=1)
            X_window[0, -1, 0] = y

        return np.array(preds).reshape(-1, 1)

    # DIRECT (multi-horizon model)
    def _predict_direct(self, X_last):

        preds = self.model.predict(X_last, verbose=0)

        # assure shape (n_steps, 1)
        return preds.reshape(-1, 1)

# Exemple d'utilisation

In [60]:
# Exemple : dataset tourisme (fictive local)

dataset = load_dataset("zaai-ai/time_series_dataset_residuals")
# Choisir le split 'train' (ou 'test')
train_split = dataset['train']

# Convertir ce split en DataFrame
df = train_split.to_pandas()

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')
df = df.set_index('Date')
df = df.groupby("Date").mean(numeric_only=True) # on aggrege les doublons

# Target
y = df['residuals'][:1000]

# proportions
train_frac = 0.6
val_frac = 0.2
test_frac = 0.2

n_total = len(y)
# print(n_total) # DEBUG

# indices de split
train_end = int(n_total * train_frac)
val_end   = int(n_total * (train_frac + val_frac))

# Splits
train = y.iloc[:train_end]
val   = y.iloc[train_end:val_end]
test  = y.iloc[val_end:]

# on normalise après le split pour éviter le leakage
mean_train = train.mean()
std_train = train.std()

train = (train - mean_train) / std_train
val   = (val - mean_train) / std_train
test  = (test - mean_train) / std_train

def regularize(series):
    return series.reindex(
        pd.date_range(series.index.min(), series.index.max(), freq="MS")
    ).interpolate()

print(df.index.duplicated().sum())

train = regularize(train)
val = regularize(val)
test = regularize(test)

train_val = pd.concat([train, val]).sort_index()

# # supprimer les doublons (si présents)
# # Vérifie les doublons
# print(train.index.duplicated().sum())
# print(val.index.duplicated().sum())
# print(train_val.index.duplicated().sum())
# print(test.index.duplicated().sum())
# train = train[~train.index.duplicated(keep='first')]
# val = val[~val.index.duplicated(keep='first')]
# train_val = train_val[~train_val.index.duplicated(keep='first')]
# test = test[~test.index.duplicated(keep='first')]

# print(train.index.is_monotonic_increasing)
# print(val.index.is_monotonic_increasing)
# print(train_val.index.is_monotonic_increasing)
# print(test.index.is_monotonic_increasing)
# # on trie par date et on réindexe en attribuant une fréquence

print(train.isna().sum())
print(val.isna().sum())
print(train_val.isna().sum())
print(test.isna().sum())

0
0
0
0
0


In [61]:
# Covariables dynamiques
X = df[['CPI', 'Inflation_Rate', 'GDP']][:1000]

# On standardise
X = (X - X.mean()) / X.std()

X_full = pd.concat([y, X], axis=1)

# Train
X_train = X_full.iloc[:train_end]

# Validation
X_val = X_full.iloc[train_end:val_end]

# Test
X_test = X.iloc[val_end:].astype(float)

# Train-Val
X_train_val = pd.concat([X_train, X_val])

# on standardise y
y = (y - y.mean()) / y.std()

data = pd.concat([y, X], axis=1)

In [62]:
y = pd.Series(y).reset_index(drop=True)
X = pd.DataFrame(X).reset_index(drop=True)

data = pd.concat([y, X], axis=1)
data.columns = ["target"] + list(X.columns)

In [63]:
def make_dataset(data, seq_len, horizon):

    values = data.values  # (time, features)

    X_past, X_future, y_out = [], [], []

    for i in range(len(values) - seq_len - horizon):

        past = values[i : i + seq_len, 1:]  # covariables
        future = values[i + seq_len : i + seq_len + horizon, 1:]
        target = values[i + seq_len : i + seq_len + horizon, 0]

        X_past.append(past)
        X_future.append(future)
        y_out.append(target)

    return (
        np.array(X_past),
        np.array(X_future),
        np.array(y_out)
    )

In [64]:
seq_len = 30
horizon = 10

X_past, X_future, y = make_dataset(data, seq_len, horizon)

In [65]:
print(X_past.shape)
print(X_future.shape)
print(y.shape)

(130, 30, 3)
(130, 10, 3)
(130, 10)


In [66]:
def make_future_sequences(X, horizon):
    X = X.values
    Xs = []

    for i in range(len(X) - horizon + 1):
        Xs.append(X[i:i+horizon])

    return np.array(Xs)

In [68]:
n_features = X_train_seq.shape[2]
wrapper = TCNRegressor(strategy="recursive")
preds = wrapper.predict(X_past, n_steps=10)

AttributeError: 'NoneType' object has no attribute 'predict'

In [48]:
n_features = X_train_seq.shape[2]

# model = build_tcn_multi_horizon(seq_len, n_features, horizon=len(test)) # multi pas
model = build_tcn(seq_len=30, n_past_features=3, horizon=10, n_future_features=3) # 1 pas
model.compile(optimizer="adam", loss="mse")

model.fit([X_past, X_future],y)

5/5 [==============================] - 2s 10ms/step - loss: 1.0020


In [51]:
predictions = model.predict([X_past, X_future])
predictions[:10]

5/5 [==============================] - 0s 3ms/step


array([[-0.22647023,  0.40354055,  0.07882409, -0.15120693,  0.23416068,
        -0.3195602 ,  0.35054496,  0.04276691,  0.05492696, -0.2040479 ],
       [-0.38172582,  0.3337524 ,  0.25275952, -0.31050467,  0.2750476 ,
        -0.47445342,  0.18619429, -0.08822823, -0.12770891, -0.25537294],
       [-0.2626024 ,  0.24173972,  0.12453476, -0.26033872,  0.30614883,
        -0.48702803,  0.05636588, -0.03899589, -0.18659443, -0.33324823],
       [-0.21732533,  0.20391363,  0.05302351, -0.24316633,  0.29509118,
        -0.46826252,  0.03984227, -0.06967265, -0.21051607, -0.35998657],
       [-0.17390484,  0.1684838 ,  0.18477958, -0.26951936,  0.4205002 ,
        -0.44739434,  0.03476793, -0.2616481 , -0.26114425, -0.41432926],
       [-0.07432864,  0.1631926 ,  0.14434443, -0.18701454,  0.46038467,
        -0.3796919 ,  0.1124167 , -0.1510826 , -0.0451219 , -0.37924862],
       [-0.26458323,  0.18016687,  0.21849343, -0.05637206,  0.36985448,
        -0.33849978,  0.27855676, -0.12809324